# Phase 0: Data Cleanup

This notebook removes uploaded documents and their vector embeddings from PostgreSQL.
Use it to reset the document store before uploading new files or to clear stale data.

It also optionally clears research session history (traces, failures).

## 1. Load Environment

In [17]:
import os
import shutil
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env") if os.path.basename(os.getcwd()) != "rhoai-custom-research-lab" else ".env"
sample_path = os.path.join(os.path.dirname(env_path), "sample.env") if os.path.dirname(env_path) else "sample.env"

if not os.path.exists(env_path):
    if os.path.exists(sample_path):
        shutil.copy(sample_path, env_path)
        print(f"Created .env from sample.env at: {env_path}")
    else:
        raise FileNotFoundError(f"Neither .env nor sample.env found. Expected at: {env_path}")

load_dotenv(env_path, override=True)
print(f"\u2705 Environment loaded from: {env_path}")

✅ Environment loaded from: /Users/hyochoi/dev/rhoai-custom-research-lab/.env


## 2. Connect to PostgreSQL

In [18]:
import psycopg2

conn = psycopg2.connect(
    host=os.getenv("PGVECTOR_HOST", "localhost"),
    port=int(os.getenv("PGVECTOR_PORT", "5432")),
    dbname=os.getenv("PGVECTOR_DB", "doc_research"),
    user=os.getenv("PGVECTOR_USER", "postgres"),
    password=os.getenv("PGVECTOR_PASSWORD", "postgres"),
)
conn.autocommit = True
cur = conn.cursor()
print("\u2705 Connected to PostgreSQL")

✅ Connected to PostgreSQL


## 3. Show Current Data Summary

Check what data is currently stored before deciding what to clean.

In [19]:
cur.execute("SELECT COUNT(*) FROM documents")
doc_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM document_chunks")
chunk_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM research_sessions")
session_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM trace_events")
trace_count = cur.fetchone()[0]

cur.execute("SELECT COUNT(*) FROM failure_log")
failure_count = cur.fetchone()[0]

print(f"\U0001F4C4 Documents:          {doc_count}")
print(f"\U0001F9E9 Document chunks:     {chunk_count}")
print(f"\U0001F50D Research sessions:   {session_count}")
print(f"\U0001F4CA Trace events:        {trace_count}")
print(f"\u26A0\uFE0F  Failure log entries: {failure_count}")

if doc_count > 0:
    cur.execute("SELECT id, name, chunk_count, status, created_at FROM documents ORDER BY created_at DESC")
    print("\n--- Documents ---")
    for row in cur.fetchall():
        print(f"  {row[1]}  (id={row[0]}, chunks={row[2]}, status={row[3]}, created={row[4]})")

📄 Documents:          0
🧩 Document chunks:     0
🔍 Research sessions:   0
📊 Trace events:        0
⚠️  Failure log entries: 0


## 4. Clear Documents and Vector Embeddings

Deletes all rows from `document_chunks` and `documents` tables.
This removes all uploaded documents and their vector embeddings.

In [20]:
cur.execute("DELETE FROM document_chunks")
deleted_chunks = cur.rowcount

cur.execute("DELETE FROM documents")
deleted_docs = cur.rowcount

print(f"\U0001F5D1\uFE0F  Deleted {deleted_chunks} chunks and {deleted_docs} documents.")
print("\u2705 Document store cleared.")

🗑️  Deleted 0 chunks and 0 documents.
✅ Document store cleared.


## 5. Clear Research Sessions (Optional)

Clears all research sessions, trace events, and failure logs.
Skip this cell if you want to keep session history.

In [21]:
cur.execute("DELETE FROM trace_events")
deleted_traces = cur.rowcount

cur.execute("DELETE FROM failure_log")
deleted_failures = cur.rowcount

cur.execute("DELETE FROM research_sessions")
deleted_sessions = cur.rowcount

print(f"\U0001F5D1\uFE0F  Deleted {deleted_traces} traces, {deleted_failures} failures, {deleted_sessions} sessions.")
print("\u2705 Session history cleared.")

🗑️  Deleted 0 traces, 0 failures, 0 sessions.
✅ Session history cleared.


## 6. Verify Cleanup

In [ ]:
cur.execute("SELECT COUNT(*) FROM documents")
print(f"Documents remaining:        {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM document_chunks")
print(f"Document chunks remaining:  {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM research_sessions")
print(f"Research sessions remaining: {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM trace_events")
print(f"Trace events remaining:     {cur.fetchone()[0]}")

cur.execute("SELECT COUNT(*) FROM failure_log")
print(f"Failure log remaining:      {cur.fetchone()[0]}")

cur.close()
conn.close()
print("\n\u2705 Cleanup complete. Database connection closed.")

Documents remaining:        0
Document chunks remaining:  0
Research sessions remaining: 0
Trace events remaining:     0
Failure log remaining:      0

✅ Cleanup complete. Database connection closed.


ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hyochoi/dev/rhoai-custom-research-lab/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users/hyochoi/dev/rhoai-custom-research-lab/.venv/lib/python3.12/site-packages/ipykernel/kernelbase.py", line 584, in shell_channel_thread_main
    _, msg2 = self.session.feed_identities(msg, copy=False)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/hyochoi/dev/rhoai-custom-research-lab/.venv/lib/python3.12/site-packages/jupyter_client/session.py", line 998, in feed_identities
    raise ValueError(msg)
ValueError: DELIM not in msg_list
ERROR:tornado.general:Uncaught exception in ZMQStream callback
Traceback (most recent call last):
  File "/Users/hyochoi/dev/rhoai-custom-research-lab/.venv/lib/python3.12/site-packages/zmq/eventloop/zmqstream.py", line 565, in _log_error
    f.result()
  File "/Users

: 